# CausalTwin — Phase 5 benchmark vs a correlational LightGBM baseline

**Thesis under test: prediction ≠ intervention.** A normal ML model can be
excellent at predicting a KPI from sensor snapshots and still be useless —
or misleading — when an operator asks *"what happens if I change this?"*

Both models below are trained on the **same** normal-operation data
(synthetic SCM, `generate(n=2000, seed=0)` — the only tier where
interventional ground truth exists) and compared on three tasks:

| Task | Question | Metric |
|---|---|---|
| (a) Intervention prediction | Predict the failure_risk change from sustained `do(fan_speed_nc)` overrides | RMSE of predicted vs actual change |
| (b) In-distribution prediction | Plain held-out prediction, no interventions (honesty check) | RMSE |
| (c) Root cause | Rank the true culprit of injected faults | Top-1 / Top-3 |

The causal entry is the Phase 4 structural simulator (own-lag AR equations +
exact steady-state propagation). The baseline is the standard "just ML"
approach: a LightGBM regressor predicting `failure_risk` from the current
state snapshot (all other nodes, incl. the lever and the pure correlate
`power_draw`). Its only what-if is the practitioner's one: bump the lever
feature and re-predict, everything else held natural.

All protocol choices (fault set, windows, seeds) are the pre-registered
Phase 4 values, fixed in `FULL_CONFIG` in the script — not tuned for this run.


In [ ]:
import sys
from pathlib import Path

# The benchmark lives in backend/scripts/benchmark_baseline.py; this
# notebook is its readable front-end and runs exactly the same code.
sys.path.insert(0, str(Path.cwd().parent / "backend" / "scripts"))
sys.path.insert(0, str(Path.cwd().parent / "backend"))

from benchmark_baseline import run_benchmark, FULL_CONFIG

print("Protocol (fixed in FULL_CONFIG):")
for key in ("n_train", "train_seed", "n_test", "test_seed",
            "fan_deltas_rpm", "scenarios", "rca_seeds"):
    print(f"  {key}: {FULL_CONFIG[key]}")


## Run the benchmark

This executes all three comparisons and writes
`backend/scripts/benchmark_results.json`. Runtime is a few seconds.

In [ ]:
results = run_benchmark()


## The comparison table

In [ ]:
import pandas as pd

rows = results["intervention"]["rows"]
table = pd.DataFrame(rows)[["scenario", "fan_delta_rpm", "actual_change",
                            "causal_predicted_change",
                            "baseline_predicted_change"]]
print(table.to_string(index=False, float_format=lambda v: f"{v:+.2f}"))

a, b, c = (results["intervention"], results["in_distribution"],
           results["root_cause"])
print(f"\nIntervention change RMSE : causal {a['causal_rmse']:.2f} | "
      f"LightGBM {a['baseline_rmse']:.2f} points")
print(f"In-distribution RMSE     : causal {b['causal_rmse']:.2f} | "
      f"LightGBM {b['baseline_rmse']:.2f} points")
print(f"Root-cause Top-1 / Top-3 : causal {c['causal']['top1']:.3f} / "
      f"{c['causal']['top3']:.3f} | importance "
      f"{c['importance']['top1']:.3f} / {c['importance']['top3']:.3f}")


## Results (measured 2026-11-26 — `backend/scripts/benchmark_results.json`)

### (a) Intervention prediction — predicted vs actual failure_risk change (points)

| Scenario | Fan change | Actual | Causal twin | LightGBM |
|---|---|---|---|---|
| normal | +40 rpm | −8.83 | −12.36 | −0.09 |
| normal | +80 rpm | −16.15 | −24.72 | −0.52 |
| normal | +120 rpm | −22.12 | −37.08 | −0.74 |
| cooling fault | +40 rpm | −10.14 | −12.36 | −0.16 |
| cooling fault | +80 rpm | −20.02 | −24.72 | −0.52 |
| cooling fault | +120 rpm | −28.84 | −37.08 | −0.77 |
| **RMSE** | | | **8.21** | **18.46** |

The LightGBM what-if is essentially *"nothing happens"* (−0.1 … −0.8 points)
while the true effect is −9 … −29 points. That is not a tuning failure: the
fan affects failure_risk only through a lagged chain
(fan → coolant → HPC temperature → risk), so a snapshot model's lever feature
carries almost no signal, and the model has no mechanism to move the
mediators. The causal twin overpredicts the magnitude (its linear steady
state vs the [0, 100]-clipped observed outcome — the same ~20–25%
observed-scale effect documented in Phase 4) but gets the direction, the
ordering and most of the size.

### (b) In-distribution prediction — held-out natural run (seed 1)

| Model | RMSE (points) |
|---|---|
| Causal (one-step structural) | **6.84** |
| LightGBM (snapshot) | 12.10 |

The causal model wins here too on this system — honestly reported, though
it is *not* the load-bearing claim: it wins because it predicts from the
true lagged parents, while the snapshot model must infer the same state
from contemporaneous correlates. On a system where the KPI's parents were
directly and contemporaneously observed, a correlational model could
match or beat it at pure prediction — and it would *still* fail task (a).

### (c) Root cause — Phase 4 fault set (4 faults × 3 seeds)

| Method | Top-1 | Top-3 |
|---|---|---|
| Causal RCA (per incident) | **1.000** | **1.000** |
| LightGBM gain importance (global) | 0.250 | 0.750 |

Importance ranking: oil_pressure > hpc_outlet_temp > vibration > load >
coolant_temp > fan_speed_nc > power_draw > maintenance_event. One global
ranking is scored against every incident — importance has no per-incident
view, so a fault at `coolant_temp` (rank 5) can never be found by it, no
matter how clean the incident is. That asymmetry is the point, and it is
how practitioners actually use importance for diagnosis.

## What this proves — and what it doesn't

**Proves (on the synthetic SCM, where truth is known):** for *interventional*
questions — what-if prediction and per-incident root-cause — the causal
twin is decisively more accurate than a standard correlational model
trained on the same data (intervention RMSE 8.21 vs 18.46 points; RCA
Top-1 1.000 vs 0.250). The baseline's failure is structural, not a matter
of hyperparameters: a snapshot regressor cannot express how an
intervention propagates.

**Doesn't prove:** anything about real machinery. CMAPSS is observational
with no interventional ground truth, so per the project's honest-claims
rule these numbers are claimed only for the synthetic tier. Nor does it
prove causal models are better *predictors* in general — task (b) is a
single system with lagged ground truth, and pure prediction was never the
claim.
